In [97]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
#from dotenv import load_dotenv; load_dotenv('C:\git-repository\Blog\.gitignore\.env')
from dotenv import load_dotenv; load_dotenv("C:\git-repository\BLS_prediction_project\.gitignore\.env")
import requests
import fredapi as fa
import seaborn as sns
import statsmodels.api as sm

<>:6: SyntaxWarning: invalid escape sequence '\g'
<>:6: SyntaxWarning: invalid escape sequence '\g'
C:\Users\paulp\AppData\Local\Temp\ipykernel_12392\1123047401.py:6: SyntaxWarning: invalid escape sequence '\g'
  from dotenv import load_dotenv; load_dotenv("C:\git-repository\BLS_prediction_project\.gitignore\.env")


#### FRED DATA WRANGLING & CLEANING

In [98]:
# NON-FARM PAYROLLS -- MONTHLY

fred_key = os.getenv('fred')
base_url = 'https://api.stlouisfed.org/fred/'
obs_endpoint = 'series/observations'

series_id = 'PAYEMS'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'm'
ts_units = 'lin'

payems_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=payems_params)
data = response.json()
payems_df = pd.DataFrame(data['observations'])
payems_df['date'] = pd.to_datetime(payems_df['date'])
payems_df['value'] = pd.to_numeric(payems_df['value'], errors='coerce')
payems_df = (
    payems_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()     # USE LAST OBSERVATION WITH PAYROLLS, AVERAGES DILUTE THE SIGNAL
    .rename(columns={'value': 'payems'})
)

In [99]:
# UNEMPLOYMENT CLAIMS -- WEEKLY

series_id = 'ICSA'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'w'
ts_units = 'lin'

unemp_claims_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=unemp_claims_params)
data = response.json()
unemp_claims_df = pd.DataFrame(data['observations'])

unemp_claims_df['date'] = pd.to_datetime(unemp_claims_df['date'])
unemp_claims_df['value'] = pd.to_numeric(unemp_claims_df['value'], errors='coerce')

unemp_claims_df = (
    unemp_claims_df
    .set_index('date')[['value']]
    .resample('ME')
    .mean()
    .rename(columns={'value': 'unemp_claims'})
)

In [105]:
# CONTINUING CLAIMS -- WEEKLY

series_id = 'CCSA'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'w'
ts_units = 'lin'

cont_claims_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=cont_claims_params)
data = response.json()
cont_claims_df = pd.DataFrame(data['observations'])

cont_claims_df['date'] = pd.to_datetime(cont_claims_df['date'])
cont_claims_df['value'] = pd.to_numeric(cont_claims_df['value'], errors='coerce')

cont_claims_df = (
    cont_claims_df
    .set_index('date')[['value']]
    .resample('ME')
    .mean()
    .rename(columns={'value': 'cont_claims'})
)

In [101]:
# MARKET YIELD ON 2-YEAR TREASURY SECURITIES -- DAILY

series_id = 'DGS2'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

two_year_yield_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=two_year_yield_params)
data = response.json()
two_year_yield_df = pd.DataFrame(data['observations'])
two_year_yield_df['date'] = pd.to_datetime(two_year_yield_df['date'])
two_year_yield_df['value'] = pd.to_numeric(two_year_yield_df['value'], errors='coerce')

two_year_yield_df = (
    two_year_yield_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()     # USE LAST OBSERVATION WITH YIELDS, AVERAGES DILUTE THE SIGNAL
    .rename(columns={'value': 'two_year_yield'})
)

In [102]:
# MARKET YIELD ON 10-YEAR TREASURY SECURITIES -- DAILY

series_id = 'DGS10'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

ten_year_yield_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=ten_year_yield_params)
data = response.json()  
ten_year_yield_df = pd.DataFrame(data['observations'])

ten_year_yield_df['date'] = pd.to_datetime(ten_year_yield_df['date'])
ten_year_yield_df['value'] = pd.to_numeric(ten_year_yield_df['value'], errors='coerce')

ten_year_yield_df = (
    ten_year_yield_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'ten_year_yield'})
)

In [103]:
# 10 YEAR TREASURY YIELD SPREAD -- DAILY

series_id = 'T10Y2Y'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

ten_year_treasury_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=ten_year_treasury_params)
data = response.json()  
ten_year_treasury_df = pd.DataFrame(data['observations'])

ten_year_treasury_df['date'] = pd.to_datetime(ten_year_treasury_df['date'])
ten_year_treasury_df['value'] = pd.to_numeric(ten_year_treasury_df['value'], errors='coerce')

ten_year_treasury_df = (
    ten_year_treasury_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'ten_year_treasury_spread'})
)

In [104]:
# UNIVERSITY OF MICHIGAN: CONSUMER SENTIMENT -- MONTHLY

series_id = 'UMCSENT'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'm'
ts_units = 'lin'

sentiment_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=sentiment_params)
data = response.json()
sentiment_df = pd.DataFrame(data['observations'])
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
sentiment_df['value'] = pd.to_numeric(sentiment_df['value'], errors='coerce')
sentiment_df = (
    sentiment_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'sentiment'})
)
sentiment_df['sentiment_lag1'] = sentiment_df['sentiment'].shift(1) # LAGGED SENTIMENT TO AVOID LOOKAHEAD BIAS IN MODELING

In [89]:
fred_data_df = (
    payems_df
    .join(unemp_claims_df, how='outer')
    .join(cont_claims_df, how='outer')  
    .join(two_year_yield_df, how='outer')
    .join(ten_year_yield_df, how='outer')
    .join(ten_year_treasury_df, how='outer')
    .join(sentiment_df, how='outer')

)

fred_data_df.describe()
#fred_data_df.head()

,payems,unemp_claims,cont_claims,two_year_yield,ten_year_yield,ten_year_treasury_spread,sentiment,sentiment_lag1
count,316.000000,3.160000e+02,3.160000e+02,317.000000,317.000000,317.000000,315.000000,314.000000
mean,140653.768987,3.693807e+05,3.039510e+06,2.266435,3.316246,1.049811,81.642222,81.732484
std,9268.015649,3.222431e+05,2.159085e+06,1.759850,1.290699,0.965067,14.086899,14.017851
min,129706.000000,1.975000e+05,1.365250e+06,0.110000,0.550000,-1.060000,50.000000,50.000000
25%,132342.250000,2.385625e+05,1.938950e+06,0.640000,2.230000,0.220000,71.150000,71.500000
50%,137748.000000,3.218750e+05,2.565500e+06,1.840000,3.440000,1.010000,82.700000,82.800000
75%,148315.000000,3.968625e+05,3.440562e+06,3.840000,4.300000,1.880000,93.000000,93.000000
max,158736.000000,4.663250e+06,2.032640e+07,6.690000,6.680000,2.840000,112.000000,112.000000


#### BLS DATA WRANGLING & CLEANING

In [117]:
import json
import prettytable
bls_key = os.getenv('BLS')

In [215]:
BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

def fetch_bls_series(series, **kwargs):
    # """
    # Pass in a list of BLS timeseries to fetch data and return the series
    # in JSON format. Arguments can also be provided as kwargs:
    #     - startyear (4 digit year)
    #     - endyear (4 digit year)
    #     - catalog (True or False)
    #     - calculations (True or False)
    #     - annualaverage (True or False)
    #     - registrationKey (api key from BLS website)
    # If the registrationKey is not passed in, this function will use the
    # BLS_API_KEY fetched from the environment.
    # """
    if len(series) < 1 or len(series) > 25:
        raise ValueError("Must pass in between 1 and 25 series ids")
    # Create headers and payload post data
    headers = {'Content-Type': 'application/json'}
    payload = {
        'seriesid': series,
        'registrationKey': bls_key,
    }
    # Update the payload with the keyword arguments and convert to JSON
    payload.update(kwargs)
    payload = json.dumps(payload)
    # Fetch the response from the BLS API
    response = requests.post(BLS_ENDPOINT, data=payload, headers=headers)
    response.raise_for_status()
    # Parse the JSON result
    #result = response.json()
    result = json.loads(response.text)
    if result['status'] != 'REQUEST_SUCCEEDED':
        raise Exception(result['message'][0])
    return result



In [ ]:
series = ['JTS000000000000000JOR', 'JTS000000000000000HIR', 'JTS000000000000000QUR', 'JTS000000000000000LDR']
start_year = 1999
end_year = 2026
json_data = fetch_bls_series(series, startyear=start_year, endyear=end_year)

In [212]:
try:
    # Create a list to store individual DataFrames
    dfs = []

    for series in json_data['Results']['series']:
        df_initial = pd.DataFrame(series)
        series_col = df_initial['seriesID'][0]

        for i in range(0, len(df_initial) - 1):
            df_row = pd.DataFrame(df_initial['data'][i])
            df_row['seriesID'] = series_col

            if 'code' not in str(df_row['footnotes']): 
                df_row['footnotes'] = ''
            else:
                df_row['footnotes'] = str(df_row['footnotes']).split("'code': '", 1)[1][:1]

            # Append df_row to the list
            dfs.append(df_row)

    # Concatenate the list of DataFrames into a single DataFrame
    df = pd.concat(dfs, ignore_index=True)

    # Save the final DataFrame to a CSV file
    df.to_csv('blsdata.csv', index=False)

except Exception as e:
    json_data['status'] == 'REQUEST_NOT_PROCESSED'
    print('BLS API has given the following Response:', json_data['status'])
    print('Reason:', json_data['message'])
    print('Error:', str(e))

# try:
#     dfs = []
#     for series in json_data['Results']['series']:
#         series_id = series['seriesID']
#         for data_point in series['data']:
#             df_row = pd.DataFrame([data_point])
#             df_row['seriesID'] = series_id
#             if 'footnotes' in df_row.columns:
#                 df_row['footnotes'] = df_row['footnotes'].apply(lambda x: x[0]['code'] if x else '')
#             dfs.append(df_row)
#     df = pd.concat(dfs, ignore_index=True)
#     df.to_csv('blsdata.csv', index=False)
# except Exception as e:
#     print('Error:', str(e))
#     if 'json_data' in locals():
#         print('Status:', json_data.get('status'))
#         print('Message:', json_data.get('message'))

In [213]:
openings_df = (
    df[df['seriesID'] == 'JTS000000000000000JOR']
    #.drop_duplicates('periodName')
    .set_index('periodName')[['value', 'year']]
    .rename(columns={'value':'job openings rate'})
)

hires_df = (
    df[df['seriesID'] == 'JTS000000000000000HIR']
    #.drop_duplicates('periodName')
    .set_index('periodName')[['value']]
    .rename(columns={'value':'hires rate'})
)

quit_df = (
    df[df['seriesID'] == 'JTS000000000000000QUR']
    #.drop_duplicates('periodName')
    .set_index('periodName')[['value']]
    .rename(columns={'value':'quit rate'})
)

discharge_df = (
    df[df['seriesID'] == 'JTS000000000000000LDR']
    #.drop_duplicates('periodName')
    .set_index('periodName')[['value']]
    .rename(columns={'value':'discharge rate'})
)

bls_01_09_df = (
    openings_df
    .join(hires_df, how='outer')
    .join(quit_df, how='outer')
    .join(discharge_df, how='outer')
)

bls_01_09_df

,job openings rate,year,hires rate,quit rate,discharge rate
periodName,,,,,
April,1.7,2009,2.9,1.3,2.0
April,1.7,2009,2.9,1.3,1.4
April,1.7,2009,2.9,1.3,1.5
April,1.7,2009,2.9,1.3,1.4
April,1.7,2009,2.9,1.3,1.5
...,...,...,...,...,...
September,3.0,2001,3.8,2.1,1.5
September,3.0,2001,3.8,2.1,1.5
September,3.0,2001,3.8,2.1,1.5


In [195]:
print(bls_df['year'].unique())

['2009' '2008' '2007' '2006' '2005' '2004' '2003' '2002' '2001']


In [216]:
series = ['JTS000000000000000JOR', 'JTS000000000000000HIR', 'JTS000000000000000QUR', 'JTS000000000000000LDR']
start_year = 2009
end_year = 2026
json_data = fetch_bls_series(series, startyear=start_year, endyear=end_year)

Exception: Request could not be serviced, as the daily threshold for total number of requests allocated to the user with registration key null has been reached.

In [ ]:
try:
    # Create a list to store individual DataFrames
    dfs = []

    for series in json_data['Results']['series']:
        df_initial = pd.DataFrame(series)
        series_col = df_initial['seriesID'][0]

        for i in range(0, len(df_initial) - 1):
            df_row = pd.DataFrame(df_initial['data'][i])
            df_row['seriesID'] = series_col

            if 'code' not in str(df_row['footnotes']): 
                df_row['footnotes'] = ''
            else:
                df_row['footnotes'] = str(df_row['footnotes']).split("'code': '", 1)[1][:1]

            # Append df_row to the list
            dfs.append(df_row)

    # Concatenate the list of DataFrames into a single DataFrame
    df = pd.concat(dfs, ignore_index=True)

    # Save the final DataFrame to a CSV file
    df.to_csv('blsdata.csv', index=False)

except Exception as e:
    json_data['status'] == 'REQUEST_NOT_PROCESSED'
    print('BLS API has given the following Response:', json_data['status'])
    print('Reason:', json_data['message'])
    print('Error:', str(e))